# Silver layer

The silver layer is build up like this:  
1.  import bronze df and check.  
2.  quick sanity checks, col present, missing val, data type.  
3.  Define doc boundary and silver schema.  
4.  one coherent text (silver_text).  
5.  inspect silver_text.  
6.  what cleaning is allowed?  
7.  start preprocessing.  
8.  spaCY annotation.  
9.  output to gold.  

In [17]:
# Import needed libraries
import pandas as pd
import spacy
import os
import re

# 1 Import bronze dataframe

In [18]:
# Load bronze data
bronze_data = pd.read_csv("../Data/bronze/bronze_data.csv")
print(bronze_data.head())

       id                                              title  \
0  doc_01  Assessing Climate Adaptation Measures in Low-L...   
1  doc_02  Evaluating Circular Economy Practices in Regio...   
2  doc_03  Data Integration Challenges in Cross-Sector Re...   
3  doc_04  Designing Indicators for Municipal Climate Res...   
4  doc_05  Improving Data Literacy Through Applied Educat...   

                                             content  
0  Introduction\nClimate adaptation has become a ...  
1  Introduction\nThe circular economy seeks to re...  
2  Introduction\nCross-sector research collaborat...  
3  Introduction\nMunicipalities play a crucial ro...  
4  Introduction\nData literacy has become an esse...  


# 2 Sanity checks

In [19]:
# Checking info about the bronze data.
print(bronze_data.info())
print("---" * 50)
print(bronze_data.describe())

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       10 non-null     str  
 1   title    10 non-null     str  
 2   content  10 non-null     str  
dtypes: str(3)
memory usage: 372.0 bytes
None
------------------------------------------------------------------------------------------------------------------------------------------------------
            id                                              title  \
count       10                                                 10   
unique      10                                                 10   
top     doc_01  Assessing Climate Adaptation Measures in Low-L...   
freq         1                                                  1   

                                                  content  
count                                                  10  
unique                                                 10  
top    

In [20]:
# checking for missing values in title and content
print(f"Missing values in title: {bronze_data['title'].isna().sum()}")
print(f"Missing values in content: {bronze_data['content'].isna().sum()}")

print("---" * 20)

# We check Data type of headings
print(f"Data type: {bronze_data['content'].map(type).value_counts().head()}")
print(f"Data type: {bronze_data['title'].map(type).value_counts().head()}")

print("---" * 20)

# We check if ID is unique
print(f"Is ID unique? {bronze_data['id'].is_unique}")

Missing values in title: 0
Missing values in content: 0
------------------------------------------------------------
Data type: content
<class 'str'>    10
Name: count, dtype: int64
Data type: title
<class 'str'>    10
Name: count, dtype: int64
------------------------------------------------------------
Is ID unique? True


# 3 Define document boundary and silver schema.  

we will have a look at what a document can be in our case.  
we got an ID, Title, and Content, the boundary is one row one document. NLP models are text based and look at documents.
not tables.  

fixing the boundary here will make sure prcessing is consistent.

Silver is meant to transform data into linguistically meaningful text, preserve semantic content, and prepare the documents for NLP models.  

So, silver has ID, raw title, raw content, and Silver text.

for reproducability we add supporting fields such as document type (media, project).  

The Silver document text is created by combining structured fields into a readable document - title -> break -> main content.  



# 4 Construct silver text  

We looked at the content of each document (id, title, content).  
For silver text we construct one line per document we dont need id for this.  
we start wil title, paragraph boundary, content, line break in content we keep, dont lowercase or remove puncrtuation.  

Then we check:  
1.  title appears at the top
2.  content starts on a new paragraph
3.  no fields are missing
4.  no unintended text duplication

For multiple documents, we verify:
1.  all documents have a Silver text
2.  average length looks reasonable
3.  no empty or broken strings

# 5 inspect and normalize text

For each document, the following elements are conceptually available:

- the document title  
- the document content  
- the intended Silver text structure (defined in Step 4)

At this stage, the text may still contain:
- inconsistent spacing
- repeated paragraphs
- boilerplate blocks
- encoding or formatting artifacts

Before applying any normalization, the text is inspected to understand its characteristics and potential issues.

Key aspects to inspect include:
- empty or near-empty documents
- unusually short or long documents
- repeated section headers
- repeated paragraphs
- visible UI or boilerplate text
- broken formatting (e.g., one word per line)

This inspection determines which normalization steps are justified and prevents unnecessary or harmful cleaning.



# 6 Silver preprocessing pre-NLP-techniques

In this layer we will actually touch the code.  
Build ONE coherent text string per document (title + content).  
Apply light normalization that improves consistency WITHOUT changing meaning.  
Remove exact duplicate paragraphs inside a document (common in exported/CMS text).  
Create simple quality flags (useful for reporting and debugging).  

In [21]:
print(bronze_data["content"].iloc[0])


Introduction
Climate adaptation has become a central topic in regional planning as coastal areas face increasing risks from sea-level rise, extreme precipitation, and prolonged droughts. Low-lying coastal regions are particularly vulnerable due to their exposure to flooding, saltwater intrusion, and land subsidence. These risks threaten not only physical infrastructure but also ecosystems, economic activities, and the livability of coastal communities. As climate impacts intensify, regional authorities are under growing pressure to implement effective adaptation measures.

Background and context
Over the past decade, a wide range of climate adaptation strategies have been proposed and implemented in coastal regions. These strategies include hard infrastructure solutions such as dikes, storm surge barriers, and pumping stations, as well as nature-based solutions like wetland restoration, dune reinforcement, and managed realignment. While the technical feasibility of many measures is wel

In [27]:
# First we inspect the bronze data, we see that id, title, and content are nicely split up.
# before any NLP steps, we need to clean the data.
# we start with the content column, we will remove any special characters, numbers, and extra spaces.

# Create clean text function for content column
def clean_text(text):
    if pd.isna(text):
        return ""
    if not isinstance(text, str):
        text = str(text)
    # ensure we are working with a string (handle unexpected Series)
    if isinstance(text, pd.Series):
        text = " ".join(text.astype(str).tolist())
    text = re.sub(r"[^A-Za-z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Apply cleaning to the entire content column and store as a new column
bronze_data["clean_content"] = bronze_data["content"].apply(clean_text)

# Show sample of cleaned content
print(bronze_data["clean_content"].head(10))

0    Introduction Climate adaptation has become a c...
1    Introduction The circular economy seeks to red...
2    Introduction Crosssector research collaboratio...
3    Introduction Municipalities play a crucial rol...
4    Introduction Data literacy has become an essen...
5    Introduction Urban heat stress has emerged as ...
6    Introduction Water quality monitoring is essen...
7    Introduction The transition from fossilbased e...
8    Introduction Research projects generate a larg...
9    Introduction Dashboards have become a common t...
Name: clean_content, dtype: str


In [31]:
# small sanity check to see if the cleaning worked as expected
print(bronze_data["content"])
print("---" * 30)
print(bronze_data["clean_content"])

0    Introduction\nClimate adaptation has become a ...
1    Introduction\nThe circular economy seeks to re...
2    Introduction\nCross-sector research collaborat...
3    Introduction\nMunicipalities play a crucial ro...
4    Introduction\nData literacy has become an esse...
5    Introduction\nUrban heat stress has emerged as...
6    Introduction\nWater quality monitoring is esse...
7    Introduction\nThe transition from fossil-based...
8    Introduction\nResearch projects generate a lar...
9    Introduction\nDashboards have become a common ...
Name: content, dtype: str
------------------------------------------------------------------------------------------
0    Introduction Climate adaptation has become a c...
1    Introduction The circular economy seeks to red...
2    Introduction Crosssector research collaboratio...
3    Introduction Municipalities play a crucial rol...
4    Introduction Data literacy has become an essen...
5    Introduction Urban heat stress has emerged as ...
6  

In [33]:
# now we do the same for the title column, we will remove any special characters, numbers, and extra spaces.
#Before:
print(bronze_data["title"].head(10))

bronze_data["clean_title"] = bronze_data["title"].apply(clean_text)

print("---" * 30)
print(bronze_data["clean_title"].head(10))

0    Assessing Climate Adaptation Measures in Low-L...
1    Evaluating Circular Economy Practices in Regio...
2    Data Integration Challenges in Cross-Sector Re...
3    Designing Indicators for Municipal Climate Res...
4    Improving Data Literacy Through Applied Educat...
5    Assessing the Impact of Nature-Based Solutions...
6    Challenges in Monitoring Water Quality Using L...
7    Stakeholder Engagement in Regional Energy Tran...
8    Developing a Knowledge Platform for Research R...
9    Assessing the Use of Dashboards for Decision S...
Name: title, dtype: str
------------------------------------------------------------------------------------------
0    Assessing Climate Adaptation Measures in LowLy...
1    Evaluating Circular Economy Practices in Regio...
2    Data Integration Challenges in CrossSector Res...
3    Designing Indicators for Municipal Climate Res...
4    Improving Data Literacy Through Applied Educat...
5    Assessing the Impact of NatureBased Solutions ...
6    

In [ ]:
# Now we see that we have 2 new columns in addition to the original content and title columns,
# we have clean_content and clean_title which are cleaned versions of the original columns.
# We can now use these cleaned columns for further processing in the silver layer.
print(bronze_data.head())

       id                                              title  \
0  doc_01  Assessing Climate Adaptation Measures in Low-L...   
1  doc_02  Evaluating Circular Economy Practices in Regio...   
2  doc_03  Data Integration Challenges in Cross-Sector Re...   
3  doc_04  Designing Indicators for Municipal Climate Res...   
4  doc_05  Improving Data Literacy Through Applied Educat...   

                                             content  \
0  Introduction\nClimate adaptation has become a ...   
1  Introduction\nThe circular economy seeks to re...   
2  Introduction\nCross-sector research collaborat...   
3  Introduction\nMunicipalities play a crucial ro...   
4  Introduction\nData literacy has become an esse...   

                                       clean_content  \
0  Introduction Climate adaptation has become a c...   
1  Introduction The circular economy seeks to red...   
2  Introduction Crosssector research collaboratio...   
3  Introduction Municipalities play a crucial rol...  

In [35]:
# We check for duplicates in the data, since its large text data, we will check for exact duplicate entries.
print(f"number of duplicated entries: {bronze_data.duplicated(subset=['clean_content', 'clean_title']).sum()}")

number of duplicated entries: 0


In [24]:
# Construct silver_text

# Define light normalization functions

# Apply cleaning to all documents

# Add quality control steps

# Santity checks